# YOLO Assignment : Rectangle & Polygon
4 คลาส : Dog, Person, Cat, Bunny

1. ตั้งค่า
2. ดูข้อมูล
3. รวมข้อมูลเป็น 4 คลาส
4. เทรน
5. ผลการเทรน
6. ทดสอบกับ test set
7. Predict
8. Export ไปใช้บนเว็บ

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import shutil
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from ultralytics import YOLO

## 1. ตั้งค่า

In [ ]:
base_dir = os.path.abspath("..")
rect_path = os.path.join(base_dir, "Dataset", "Dataset_Rectangle", "Dataset_Rectangle")
poly_path = os.path.join(base_dir, "Dataset", "Dataset_Polygon")
weight_dir = os.path.join(base_dir, "Weights")
result_dir = os.path.join(base_dir, "ผลการเทรน")
predict_dir = os.path.join(base_dir, "Predict")

if not os.path.exists(os.path.join(poly_path, "person and dog")):
    shutil.unpack_archive(os.path.join(poly_path, "Dataset_Polygon.zip"), poly_path)

class_names = ["Dog", "Person", "Cat", "Bunny"]
colors = [(255, 140, 0), (0, 90, 255), (200, 0, 150), (0, 170, 60)]

datasets = {
    "rectangle": {
        "source": rect_path,
        "folders": [("person and dog", "dp_", {0: 0, 1: 1}), ("cat and bunny", "cb_", {0: 3, 1: 2})],
        "model": "yolo11n.pt",
    },
    "polygon": {
        "source": poly_path,
        "folders": [("person and dog", "dp_", {0: 0, 1: 1}), ("cat and buuny", "cb_", {0: 3, 1: 2})],
        "model": "yolo11n-seg.pt",
    },
}

device = 0 if torch.cuda.is_available() else "cpu"
print("device =", device, torch.cuda.get_device_name(0) if device == 0 else "")

## 2. ดูข้อมูล

In [ ]:
for name, d in datasets.items():
    for folder, prefix, new_id in d["folders"]:
        classes = open(os.path.join(d["source"], folder, "classes.txt")).read().split()
        print(f"{name:10s} {folder:15s} {classes}")

In [ ]:
rect_sample = os.path.join(rect_path, "person and dog", "train")
poly_sample = os.path.join(poly_path, "cat and buuny", "train")

print(open(rect_sample + "/labels/82.txt").read())
print(open(poly_sample + "/labels/76.txt").read()[:200])

In [ ]:
def imread(path):
    return cv2.imdecode(np.fromfile(path, dtype=np.uint8), cv2.IMREAD_COLOR)


def draw_label(img_file, label_file, names):
    img = cv2.cvtColor(imread(img_file), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    overlay = img.copy()
    texts = []

    for line in open(label_file).read().splitlines():
        p = line.split()
        if len(p) < 5:
            continue
        c = int(p[0])
        color = colors[c]
        values = np.array(p[1:], dtype=float)

        if len(values) == 4:
            x, y, bw, bh = values * [w, h, w, h]
            x1, y1 = int(x - bw / 2), int(y - bh / 2)
            x2, y2 = int(x + bw / 2), int(y + bh / 2)
            cv2.rectangle(img, (x1, y1), (x2, y2), color, 3)
        else:
            pts = (values.reshape(-1, 2) * [w, h]).astype(np.int32)
            cv2.fillPoly(overlay, [pts], color)
            cv2.polylines(img, [pts], True, color, 3)
            x1, y1 = pts.min(axis=0)
        texts.append((names[c], int(x1), int(y1), color))

    img = cv2.addWeighted(overlay, 0.35, img, 0.65, 0)
    for text, x1, y1, color in texts:
        cv2.putText(img, text, (x1, max(y1 - 8, 20)), cv2.FONT_HERSHEY_SIMPLEX, 1, color, 2)
    return img

In [ ]:
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.imshow(draw_label(rect_sample + "/images/82.jpg", rect_sample + "/labels/82.txt", ["Dog", "Person"]))
plt.title("Rectangle")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(draw_label(poly_sample + "/images/76.jpg", poly_sample + "/labels/76.txt", ["Bunny", "Cat"]))
plt.title("Polygon")
plt.axis("off")

plt.show()

## 3. รวมข้อมูลเป็น 4 คลาส
Dog = 0, Person = 1, Cat = 2, Bunny = 3

In [ ]:
def merge_dataset(name):
    d = datasets[name]
    out_path = f"datasets/{name}"
    shutil.rmtree(out_path, ignore_errors=True)
    count = {}

    for folder, prefix, new_id in d["folders"]:
        for split in ["train", "val", "test"]:
            img_dir = os.path.join(d["source"], folder, split, "images")
            lbl_dir = os.path.join(d["source"], folder, split, "labels")
            os.makedirs(f"{out_path}/{split}/images", exist_ok=True)
            os.makedirs(f"{out_path}/{split}/labels", exist_ok=True)

            for file in os.listdir(img_dir):
                stem = os.path.splitext(file)[0]
                shutil.copy(f"{img_dir}/{file}", f"{out_path}/{split}/images/{prefix}{file}")

                lines = []
                lbl_file = f"{lbl_dir}/{stem}.txt"
                if os.path.exists(lbl_file):
                    for line in open(lbl_file).read().splitlines():
                        p = line.split()
                        if len(p) < 5:
                            continue
                        c = new_id[int(p[0])]
                        lines.append(" ".join([str(c)] + p[1:]))
                        count[(split, c)] = count.get((split, c), 0) + 1

                with open(f"{out_path}/{split}/labels/{prefix}{stem}.txt", "w") as f:
                    f.write("\n".join(lines))

    yaml_text = f"path: {os.path.abspath(out_path)}\ntrain: train/images\nval: val/images\ntest: test/images\nnames:\n"
    yaml_text += "".join(f"  {i}: {n}\n" for i, n in enumerate(class_names))
    with open(f"{out_path}/data.yaml", "w", encoding="utf-8") as f:
        f.write(yaml_text)

    table = pd.DataFrame(
        [[count.get((s, c), 0) for s in ["train", "val", "test"]] for c in range(len(class_names))],
        index=class_names, columns=["train", "val", "test"])
    table["total"] = table.sum(axis=1)
    return table

In [ ]:
tables = {name: merge_dataset(name) for name in datasets}

for name, table in tables.items():
    images = {s: len(os.listdir(f"datasets/{name}/{s}/images")) for s in ["train", "val", "test"]}
    print(name, "images =", images)
    print(table, "\n")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for i, (name, table) in enumerate(tables.items()):
    table[["train", "val", "test"]].plot(kind="bar", stacked=True, ax=ax[i], title=name)
    ax[i].set_ylabel("objects")
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
for i, name in enumerate(datasets):
    folder = f"datasets/{name}/train"
    files = [f for f in sorted(os.listdir(folder + "/images"))
             if f.startswith("cb_") and os.path.getsize(f"{folder}/labels/{os.path.splitext(f)[0]}.txt") > 0]
    stem = os.path.splitext(files[0])[0]
    plt.subplot(1, 2, i + 1)
    plt.imshow(draw_label(f"{folder}/images/{files[0]}", f"{folder}/labels/{stem}.txt", class_names))
    plt.title(name)
    plt.axis("off")
plt.show()

## 4. เทรน

In [ ]:
YOLO("yolo11n.pt").info()
YOLO("yolo11n-seg.pt").info()

In [ ]:
def train(name, epochs=60):
    model = YOLO(datasets[name]["model"])
    model.train(
        data=f"datasets/{name}/data.yaml",
        epochs=epochs,
        imgsz=640,
        batch=16,
        device=device,
        workers=0,
        patience=20,
        mosaic=1.0,
        fliplr=0.5,
        project=result_dir,
        name=name,
        exist_ok=True,
    )
    os.makedirs(weight_dir, exist_ok=True)
    shutil.copy(os.path.join(result_dir, name, "weights", "best.pt"), os.path.join(weight_dir, f"{name}_best.pt"))

In [ ]:
train("rectangle")

In [ ]:
train("polygon")

## 5. ผลการเทรน

In [ ]:
def plot_results(name):
    r = pd.read_csv(f"{result_dir}/{name}/results.csv")
    r.columns = r.columns.str.strip()
    loss_cols = [c for c in r.columns if c.endswith("_loss") and "dfl" not in c]
    map_cols = [c for c in r.columns if c.startswith("metrics/mAP")]

    plt.figure(figsize=(14, 4))
    plt.subplot(1, 2, 1)
    for c in loss_cols:
        plt.plot(r[c], label=c)
    plt.xlabel("epoch")
    plt.title(f"{name} : loss")
    plt.legend(fontsize=8)

    plt.subplot(1, 2, 2)
    for c in map_cols:
        plt.plot(r[c], label=c)
    plt.xlabel("epoch")
    plt.title(f"{name} : mAP")
    plt.legend(fontsize=8)
    plt.show()


plot_results("rectangle")
plot_results("polygon")

In [ ]:
plt.figure(figsize=(16, 8))
for i, name in enumerate(datasets):
    plt.subplot(1, 2, i + 1)
    plt.imshow(plt.imread(f"{result_dir}/{name}/confusion_matrix_normalized.png"))
    plt.title(name)
    plt.axis("off")
plt.show()

## 6. ทดสอบกับ test set

In [ ]:
rect_model = YOLO(os.path.join(weight_dir, "rectangle_best.pt"))
poly_model = YOLO(os.path.join(weight_dir, "polygon_best.pt"))

rect_test = rect_model.val(data="datasets/rectangle/data.yaml", split="test", workers=0,
                           project=result_dir, name="rectangle_test", exist_ok=True)
poly_test = poly_model.val(data="datasets/polygon/data.yaml", split="test", workers=0,
                           project=result_dir, name="polygon_test", exist_ok=True)

In [ ]:
summary = pd.DataFrame({
    "Rectangle (box)": [rect_test.box.map50, rect_test.box.map],
    "Polygon (box)": [poly_test.box.map50, poly_test.box.map],
    "Polygon (mask)": [poly_test.seg.map50, poly_test.seg.map],
}, index=["mAP50", "mAP50-95"])
summary.round(3)

In [ ]:
per_class = pd.DataFrame({
    "Rectangle": dict(zip(rect_test.ap_class_index, rect_test.box.ap50)),
    "Polygon (mask)": dict(zip(poly_test.ap_class_index, poly_test.seg.ap50)),
})
per_class.index = [class_names[i] for i in per_class.index]
per_class.round(3)

## 7. Predict

In [ ]:
rect_pred = rect_model.predict("datasets/rectangle/test/images", conf=0.35, save=True, verbose=False,
                               project=predict_dir, name="predict_rectangle", exist_ok=True)
poly_pred = poly_model.predict("datasets/polygon/test/images", conf=0.35, save=True, verbose=False,
                               project=predict_dir, name="predict_polygon", exist_ok=True)

plt.figure(figsize=(16, 8))
for i in range(4):
    plt.subplot(2, 4, i + 1)
    plt.imshow(rect_pred[i].plot()[:, :, ::-1])
    plt.title("Rectangle")
    plt.axis("off")

    plt.subplot(2, 4, i + 5)
    plt.imshow(poly_pred[i].plot()[:, :, ::-1])
    plt.title("Polygon")
    plt.axis("off")
plt.show()

In [ ]:
r = poly_pred[0]
for box, poly in zip(r.boxes, r.masks.xy):
    print(class_names[int(box.cls)], "conf =", round(float(box.conf), 2),
          "box =", box.xyxy[0].int().tolist(), "polygon =", len(poly), "points")

## 8. Export ไปใช้บนเว็บ

In [ ]:
onnx_file = poly_model.export(format="onnx", imgsz=640, simplify=True, opset=17)
print(onnx_file)

web_model = os.path.join(base_dir, "web", "model")
os.makedirs(web_model, exist_ok=True)
shutil.copy(onnx_file, os.path.join(web_model, "best.onnx"))
print("copy to", web_model)